In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

In [4]:
path = kagglehub.dataset_download("surajghuwalewala/ham1000-segmentation-and-classification")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ham1000-segmentation-and-classification' dataset.
Path to dataset files: /kaggle/input/ham1000-segmentation-and-classification


In [5]:
def create_linear_kernel(length, angle_degrees):
    """
    Creates a linear structuring element of a specified length and angle for morphological operations.

    Parameters:
    - length: odd integer (e.g., 9, 11, 15). Defines the size of the matrix.
    - angle_degrees: float or int (0-180). Angle of the line in degrees.
    """
    kernel = np.zeros((length, length), dtype=np.uint8)
    center = length // 2
    angle_rad = np.radians(angle_degrees)
    x_offset = int(np.round((length / 2) * np.cos(angle_rad)))
    y_offset = int(np.round((length / 2) * np.sin(angle_rad)))
    p1 = (center - x_offset, center - y_offset)
    p2 = (center + x_offset, center + y_offset)
    cv2.line(kernel, p1, p2, 1, 1)
    
    return kernel

In [ ]:
def filter_mask_with_opening(rough_mask, kernel_length=11):
    """Uses directional opening to clean the rough mask of hair pixels, preserving circular features like globules."""
    clean_mask = np.zeros_like(rough_mask)
    angles = [0, 30, 60, 90, 120, 150]
    for angle in angles:
        kernel = create_linear_kernel(kernel_length, angle)
        opened_mask = cv2.morphologyEx(rough_mask, cv2.MORPH_OPEN, kernel)
        clean_mask = cv2.max(clean_mask, opened_mask)
    
    return clean_mask

In [ ]:
def hair_removal(img_rgb):
    """
    Remove hair from the input RGB image using morphological operations and inpainting.
    """
    cross_kernel_length = 15
    opening_kernel_length = 15
    threshold_value = 15
    max_value = 255
    dilate_kernel_length = 3
    inpaint_radius = 3

    r_channel = img_rgb[:, :, 0]

    kernel_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (cross_kernel_length, cross_kernel_length))
    blackhat = cv2.morphologyEx(r_channel, cv2.MORPH_BLACKHAT, kernel_cross)

    _, rough_mask = cv2.threshold(blackhat, threshold_value, max_value, cv2.THRESH_BINARY)
    clean_mask = filter_mask_with_opening(rough_mask, kernel_length=opening_kernel_length)

    kernel_dilate = np.ones((dilate_kernel_length, dilate_kernel_length), np.uint8)
    mask_dilated = cv2.dilate(clean_mask, kernel_dilate, iterations=1)
    
    inpainted = cv2.inpaint(img_rgb, mask_dilated, inpaintRadius=inpaint_radius, flags=cv2.INPAINT_NS)

    return mask_dilated, inpainted